# 3.2.2 · Storytelling con Plotly y Streamlit — DataRetail 360
**IDY1102 · Unidad 3 · Experiencia 3 · Actividad 3.2**

| Campo | Detalle |
|---|---|
| **RA** | RA3 — Construye dashboards e informes para la toma de decisiones |
| **IL** | IL 3.3 (visualización interactiva con Python) · IL 3.4 (storytelling) |
| **Datos** | KPIs **reales** de la ficha 3.1.3, leídos desde el Data Warehouse (capa Gold) |
| **Entregable** | 3 gráficos con título-mensaje + narrativa SCI con acción (+ app Streamlit opcional) |

> Los datos son **sintéticos** (los genera `scripts/generar_datos_ejemplo.py`) y la historia está diseñada en ellos;
> en el informe hay que decirlo.

## 0. Requisitos
1. El DW cargado: `python run_pipeline.py` (notebooks 01 a 06).
2. Las librerías de esta actividad: `pip install -r requirements-storytelling.txt`.

La lectura de datos, los gráficos y la narrativa viven en `storytelling/kpis.py` para que **este notebook y la app Streamlit cuenten
exactamente la misma historia**. Ningún número de un título o de la narrativa está escrito a mano: sale de los datos.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RAIZ = Path.cwd().parent if Path.cwd().name == "storytelling" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from storytelling import kpis as K

engine = K.get_engine()
pd.set_option("display.width", 160)

## 1. Datos: los KPIs de la ficha, desde el DWH
| KPI | Fuente | Qué calculamos |
|---|---|---|
| **K1** venta neta mensual | `fact_ventas` | venta sin devoluciones por mes y su meta (promedio de los 3 meses previos) |
| **K2** ticket promedio | `fact_ventas` | venta neta ÷ transacciones |
| **K4** quiebre de stock | `fact_inventario` | % de producto × sucursal × día con stock 0, por semana y categoría |
| **K5** efectividad de campaña | `fact_campanas` | respuesta, conversión y ROAS por campaña |

In [ ]:
k1 = K.cargar_k1_k2(engine)
k4 = K.cargar_k4(engine)
k5 = K.cargar_k5(engine)
display(k1[["mes", "venta_mm", "meta_mm", "ratio", "estado", "ticket", "ticket_var"]].round(2))
display(k5[["campana", "contactados", "respuesta_pct", "conversion_pct", "roas", "estado"]].round(2))

## 2. Gráfico 1 · K1: la venta neta frente a su meta
**Qué decisión de diseño aplica** (guías 3.1.2 y 3.1.4):
- **Línea** porque es evolución en el tiempo; la meta y el umbral de alerta se dibujan en el mismo gráfico (**contexto**: un número solo no dice nada).
- El título **es el mensaje**: dice qué pasó y cuánto, no «Venta neta mensual».
- El mes en alerta se marca con una **✕ y una ▼ en la etiqueta**, no solo con el color: se lee igual en escala de grises o con daltonismo.

In [ ]:
fig1 = K.fig_k1(k1)
fig1.show()

## 3. Gráfico 2 · K4: el quiebre de stock por semana
- Se compara **todas las categorías** (gris) contra la categoría destacada (navy): la complicación se ve porque una sube y la otra no.
- Las dos metas llevan **texto** («Meta ≤ 5 %», «Alerta > 10 %») y **tipo de línea distinto** (guiones y puntos).
- La semana del pico se **anota directamente** en el gráfico.

In [ ]:
fig2 = K.fig_k4(k4, destacar="Tecnología")
fig2.show()

## 4. Gráfico 3 · K5: respuesta, conversión y retorno por campaña
- Dos paneles que comparten las campañas: a la izquierda **cuántos respondieron y cuántos compraron**, a la derecha **el ROAS**.
- Barras **horizontales** porque los nombres de campaña son largos, **ordenadas** de mejor a peor retorno.
- Cada ROAS lleva **su estado escrito** («0,81x · Alerta»): el color (teal, dorado, naranja) refuerza pero no es lo único que informa.

In [ ]:
fig3 = K.fig_k5(k5)
fig3.show()

## 5. La narrativa SCI que une los gráficos
**Situación → Complicación → Idea.** Los tres gráficos son la **evidencia** de esta historia. El texto se arma con las cifras de los datos:

In [ ]:
sci = K.narrativa_sci(k1, k4, k5)
display(Markdown(f"**Situación.** {sci['situacion']}\n\n**Complicación.** {sci['complicacion']}\n\n**Idea.** {sci['idea']}"))

## 6. Verificación: lo que afirma la narrativa está en los datos
Si algún día cambian los datos y la historia deja de ser cierta, esta celda falla con un mensaje claro en vez de dejar un informe con afirmaciones falsas.

In [ ]:
for hecho in K.verificar_historia(k1, k4, k5):
    print("✔", hecho)

## 7. Exportar los gráficos para el informe y la presentación
Se guardan en `storytelling/salida/`. El HTML conserva la interactividad (necesita internet para cargar Plotly); el PNG requiere Chrome
(`plotly_get_chrome`, una sola vez).

In [ ]:
salida = RAIZ / "storytelling" / "salida"
salida.mkdir(exist_ok=True)
for nombre, fig in {"k1_venta_neta": fig1, "k4_quiebre": fig2, "k5_campanas": fig3}.items():
    fig.write_html(salida / f"{nombre}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(salida / f"{nombre}.png", width=1100, scale=2)
    except Exception:
        print(f"PNG de {nombre} no disponible (kaleido necesita Chrome: ejecuta `plotly_get_chrome`). El HTML sí se guardó.")
print("Archivos en", salida)

## 8. (Opcional) La misma historia como app con Streamlit
Streamlit convierte un script de Python en una app web con filtros. Desde la raíz del repo:

```bash
streamlit run storytelling/app.py
```

La app (`storytelling/app.py`) muestra cuatro tarjetas (K1, K2, K4, K5, cada una con su meta escrita), los tres gráficos y la narrativa SCI,
con tres filtros: **rango de meses** (K1 y K2), **categoría a destacar** (K4) y **campañas** (K5). Streamlit no corre dentro del notebook
(necesita su propio servidor).

## 9. Decisiones y límites (completar con el equipo)
- **Datos sintéticos con historia diseñada:** la coincidencia entre quiebre de Tecnología, caída de venta y bajo ROAS está construida en el generador. Con datos reales habría que probar la relación (por ejemplo, comparar semanas con y sin quiebre), no solo mostrarla.
- **Correlación, no causa:** la narrativa dice que los hechos «coinciden» y propone una acción; no demuestra causalidad.
- **K3 y K6 no están en esta historia:** K3 recién tiene una ventana de 90 días completa desde junio, y K6 sigue como candidato sin umbral.
- **Umbrales provisionales:** son los de la ficha 3.1.3 y se calibraron con estos datos.
- ¿Qué otro gráfico habría sido un mejor argumento para la Idea y por qué? (completar)